In [ ]:
from helper_GAN import *

I moved classes and functions from the 2X training notebook to the `helper_GAN.py` not to rewrite the same code

## X4 Scaling

In [ ]:
valid_ds = SRResNet_Dataset(HR_valid_paths, 4, ram_limit_gb=1)

In [ ]:
train_ds = SRResNet_Dataset(HR_train_paths, 4, ram_limit_gb=8)

### Day 1

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

dis_loss_fn = nn.BCELoss()
pixel_loss_fn = nn.MSELoss()

In [ ]:
checkpoint = torch.load('../model_checkpoints/SRResNet/X4.pth')
generator = SRResNet(4).to(device)
generator.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
discriminator = Discriminator().to(device)

In [ ]:
generator_opt = Adam(generator.parameters(), lr=1e-4)
discriminator_opt = Adam(discriminator.parameters(), lr=1e-4)

generator_scheduler = StepLR(generator_opt, step_size=800, gamma=0.5)
discriminator_scheduler = StepLR(discriminator_opt, step_size=800, gamma=0.5)

In [ ]:
train(generator, discriminator, train_dl, valid_dl, generator_opt, discriminator_opt, generator_scheduler, discriminator_scheduler, dis_loss_fn, pixel_loss_fn, 2000)

In [ ]:
generator.eval();

In [ ]:
targets = [X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x4 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=5):
    metrics_x4.loc[len(metrics_x4)] = calc_metrics(generator, target_ds, 4)

metrics_x4.index = [
    "31px -> 124px", "63px -> 252px", "127px -> 508px", 
    "255px -> 1020px", "510px -> 2040px"
]
metrics_x4

### Day 2

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

dis_loss_fn = nn.BCELoss()
pixel_loss_fn = nn.MSELoss()

In [ ]:
checkpoint = torch.load('../tmp_model_checkpoints/last.pth')

In [ ]:
generator = SRResNet(4).to(device)
generator.load_state_dict(checkpoint['generator_state_dict'])

discriminator = Discriminator().to(device)
discriminator.load_state_dict(checkpoint['discriminator_state_dict'])

In [ ]:
generator_opt = Adam(generator.parameters(), lr=1e-4)
generator_opt.load_state_dict(checkpoint['generator_optimizer_state_dict'])
discriminator_opt = Adam(discriminator.parameters(), lr=1e-4)
discriminator_opt.load_state_dict(checkpoint['discriminator_optimizer_state_dict'])

generator_scheduler = StepLR(generator_opt, step_size=800, gamma=0.5)
generator_scheduler.load_state_dict(checkpoint['generator_scheduler_state_dict'])
discriminator_scheduler = StepLR(discriminator_opt, step_size=800, gamma=0.5)
discriminator_scheduler.load_state_dict(checkpoint['discriminator_scheduler_state_dict'])

In [ ]:
train(generator, discriminator, train_dl, valid_dl, generator_opt, discriminator_opt, generator_scheduler, discriminator_scheduler, dis_loss_fn, pixel_loss_fn, 4000)

In [ ]:
generator.eval();

In [ ]:
targets = [X16_valid_paths, X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x4 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=5):
    metrics_x4.loc[len(metrics_x4)] = calc_metrics(generator, target_ds, 4)

metrics_x4.index = [
    "31px -> 124px", "63px -> 252px", "127px -> 508px", 
    "255px -> 1020px", "510px -> 2040px"
]
metrics_x4

### Super-resolution showcase

In [ ]:
print("31px -> 124px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((generator(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/31px', exist_ok=True)
img.save('../image_results/4X/31px/SRGAN_31px.png')
img

In [ ]:
print("63px -> 252px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((generator(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/63px', exist_ok=True)
img.save('../image_results/4X/63px/SRGAN_63px.png')
img

In [ ]:
print("127px -> 508px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((generator(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/127px', exist_ok=True)
img.save('../image_results/4X/127px/SRGAN_127px.png')
img

In [ ]:
print("255px -> 1020px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((generator(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/255px', exist_ok=True)
img.save('../image_results/4X/255px/SRGAN_255px.png')
img

In [ ]:
print("510px -> 2040px")
inp = transform(Image.open(X4_valid_paths[4])).to(device)
with torch.inference_mode():
    out = (((generator(inp[None])+1.0)/2.0).clamp(0.0, 1.0) * 255.0).squeeze()

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/4X/510px', exist_ok=True)
img.save('../image_results/4X/510px/SRGAN_510px.png')
img